# Предобработка данных для DL-модели

## Загружаем датасет

In [372]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plot
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder, LabelEncoder, MinMaxScaler

In [373]:
df = pd.read_csv('task_1_to_DL.csv')
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 89740 entries, 0 to 89739
Data columns (total 19 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   Unnamed: 0        89740 non-null  int64  
 1   popularity        89740 non-null  int64  
 2   duration_ms       89740 non-null  int64  
 3   explicit          89740 non-null  bool   
 4   danceability      89740 non-null  float64
 5   energy            89740 non-null  float64
 6   key               89740 non-null  int64  
 7   loudness          89740 non-null  float64
 8   mode              89740 non-null  int64  
 9   speechiness       89740 non-null  float64
 10  acousticness      89740 non-null  float64
 11  instrumentalness  89740 non-null  float64
 12  liveness          89740 non-null  float64
 13  valence           89740 non-null  float64
 14  tempo             89740 non-null  float64
 15  time_signature    89740 non-null  int64  
 16  track_genre       89740 non-null  object

## Начинаем обработку

**1. Обработка категориальных признаков**

In [374]:
df.select_dtypes(exclude='number').columns.to_list()

['explicit', 'track_genre', 'artist_1']

In [375]:
df['explicit']

0        False
1        False
2        False
3        False
4        False
         ...  
89735    False
89736    False
89737    False
89738     True
89739    False
Name: explicit, Length: 89740, dtype: bool

In [376]:
df['explicit'].nunique()

2

In [377]:
df['explicit'].unique()

array([False,  True])

Булева переменная - применим кодирование:
- 0 - False
- 1 - True

Достаточно применить тип данных int к столбцу.

In [378]:
df['new_explicit'] = df['explicit'].astype(int)
df['new_explicit']

0        0
1        0
2        0
3        0
4        0
        ..
89735    0
89736    0
89737    0
89738    1
89739    0
Name: new_explicit, Length: 89740, dtype: int64

**2. Разделим датасет**

Перед дальнейшей обработкой разделим выборку данных на тренировочкую, валидационную и тестовую:
- train
- val
- test

In [379]:
df_train, else_part = train_test_split(df, test_size = 0.3, random_state = 42)
df_val, df_test = train_test_split(else_part, test_size = 0.5, random_state = 42)

print(f'Размер изначального датасета: {len(df)}')

print(f'Размер тренировочного датасета: {len(df_train)}')
print(f'Размер валидационного датасета: {len(df_val)}')
print(f'Размер тестового датасета: {len(df_test)}')

Размер изначального датасета: 89740
Размер тренировочного датасета: 62818
Размер валидационного датасета: 13461
Размер тестового датасета: 13461


**3. Теперь работаем с жанрами**

In [380]:
df['track_genre']

0         acoustic
1         acoustic
2         acoustic
3         acoustic
4         acoustic
           ...    
89735       indian
89736       latino
89737    synth-pop
89738    metalcore
89739         rock
Name: track_genre, Length: 89740, dtype: object

In [381]:
df['track_genre'].nunique()

114

Всего датасет содержит 114 уникальных жанров треков. Вполне допустимо для применения OHE.

Используем OHE из sklearn: https://www.geeksforgeeks.org/machine-learning/ml-one-hot-encoding/

In [382]:
ohe_object = OneHotEncoder(handle_unknown = 'ignore', sparse_output = False)

ohe_object.fit(df_train[['track_genre']])

,categories,'auto'
,drop,None
,sparse_output,False
,dtype,<class 'numpy.float64'>
,handle_unknown,'ignore'
,min_frequency,None
,max_categories,None
,feature_name_combiner,'concat'


In [383]:
tmp_train_encoded = ohe_object.transform(df_train[['track_genre']])
tmp_val_encoded = ohe_object.transform(df_val[['track_genre']])
tmp_test_encoded = ohe_object.transform(df_test[['track_genre']])

In [384]:
df_train_encoded = pd.DataFrame(
    tmp_train_encoded,
    columns = ohe_object.get_feature_names_out(['track_genre']), 
    index = df_train.index
)

df_val_encoded = pd.DataFrame(
    tmp_val_encoded,
    columns = ohe_object.get_feature_names_out(['track_genre']),
    index = df_val.index
)

df_test_encoded = pd.DataFrame(
    tmp_test_encoded,
    columns = ohe_object.get_feature_names_out(['track_genre']),
    index = df_test.index
)


In [385]:
df_train.drop(columns = 'track_genre', inplace = True)
df_val.drop(columns = 'track_genre', inplace = True)
df_test.drop(columns = 'track_genre', inplace = True)

In [386]:
df_train = pd.concat(
    [df_train, df_train_encoded],
    axis = 1
)

df_val = pd.concat(
    [df_val, df_val_encoded],
    axis = 1
)

df_test = pd.concat(
    [df_test, df_test_encoded],
    axis = 1
)

In [387]:
df_train.columns

Index(['Unnamed: 0', 'popularity', 'duration_ms', 'explicit', 'danceability',
       'energy', 'key', 'loudness', 'mode', 'speechiness',
       ...
       'track_genre_spanish', 'track_genre_study', 'track_genre_swedish',
       'track_genre_synth-pop', 'track_genre_tango', 'track_genre_techno',
       'track_genre_trance', 'track_genre_trip-hop', 'track_genre_turkish',
       'track_genre_world-music'],
      dtype='object', length=133)

In [388]:
df_train.drop(columns = 'Unnamed: 0', inplace = True)
df_val.drop(columns = 'Unnamed: 0', inplace = True)
df_test.drop(columns = 'Unnamed: 0', inplace = True)

In [389]:
df_train.columns

Index(['popularity', 'duration_ms', 'explicit', 'danceability', 'energy',
       'key', 'loudness', 'mode', 'speechiness', 'acousticness',
       ...
       'track_genre_spanish', 'track_genre_study', 'track_genre_swedish',
       'track_genre_synth-pop', 'track_genre_tango', 'track_genre_techno',
       'track_genre_trance', 'track_genre_trip-hop', 'track_genre_turkish',
       'track_genre_world-music'],
      dtype='object', length=132)

**4. Обработаем признак - исполнитель**

Поле кодирует псевдоним артиста

In [390]:
df['artist_1'].isnull().sum()

0

Нет пропусков в имене исполнителя.

In [391]:
df[df['artist_1'] == 'Unknow']

,Unnamed: 0,popularity,duration_ms,explicit,danceability,energy,key,loudness,mode,speechiness,acousticness,instrumentalness,liveness,valence,tempo,time_signature,track_genre,artist_1,artists_amount,new_explicit


Исполниетелей со статусом 'Unknow' тоже нет.

Посмотрим на кол-во уникальных значений.

In [392]:
df['artist_1'].nunique()

17648

Вводить кодировку через OHE создаст больше 17 тыс столбцов - неоправданное раздутие датасета, поэтому будет использовать LabelEncoding.

In [393]:
label_encoding_obj = LabelEncoder()

label_encoding_obj.fit(df_train['artist_1'])

LabelEncoder()

In [394]:
artist_new_labels = {artist: label for label, artist in enumerate(label_encoding_obj.classes_)}

miss_class = len(label_encoding_obj.classes_)
miss_class

15215

In [395]:
all_artists = miss_class + 1
all_artists

15216

In [396]:
df_train['artist_label'] = df_train['artist_1'].map(artist_new_labels).fillna(miss_class).astype(int)
df_val['artist_label'] = df_val['artist_1'].map(artist_new_labels).fillna(miss_class).astype(int)
df_test['artist_label'] = df_test['artist_1'].map(artist_new_labels).fillna(miss_class).astype(int)

In [397]:
df_train[['artist_label', 'artist_1']]

,artist_label,artist_1
54346,12507,Sunnery James & Ryan Marciano
46046,2781,Crowd Lu
9222,9202,Mr. Fingers
59073,10486,Porsche Love
58039,14174,Vladimir Troshin
...,...,...
6265,7159,Krafty Kuts
54886,13034,The Doors
76820,7378,Lamb of God
860,10430,Plastilina Mosh


In [398]:
df_train.drop(columns = 'artist_1', inplace = True)
df_val.drop(columns = 'artist_1', inplace = True)
df_test.drop(columns = 'artist_1', inplace = True)

## Обработка числовых признаков.

Надо очистить данные от выбросов и провести нормализацию.

**1. Очистка выбросов**

Как было видно на boxplot - в данных много выбросов (значения, которые выходят за пределы усов), то есть значений, которые дальше чем 1,5 IQR. Самым правильным решением будет очистить датасет от таких данных.

In [399]:
df.drop(columns = 'Unnamed: 0', inplace = True)

number_columns = df.select_dtypes(include='number').columns.to_list()
number_columns

['popularity',
 'duration_ms',
 'danceability',
 'energy',
 'key',
 'loudness',
 'mode',
 'speechiness',
 'acousticness',
 'instrumentalness',
 'liveness',
 'valence',
 'tempo',
 'time_signature',
 'artists_amount',
 'new_explicit']

Исключим 'popularity' и 'artists_amount' из списка, как целевую переменную и искусственно созданную переменную без выбросов (кол-во артистов).

In [400]:
number_columns.remove('popularity')
number_columns.remove('artists_amount')
number_columns

['duration_ms',
 'danceability',
 'energy',
 'key',
 'loudness',
 'mode',
 'speechiness',
 'acousticness',
 'instrumentalness',
 'liveness',
 'valence',
 'tempo',
 'time_signature',
 'new_explicit']

Метод .clip(): для того чтобы обрезать данные ниже и выше выбросов - такие значения заменяются на крайнее минимальное и максималльное значение в полтора межквантильных размаха.

https://www.geeksforgeeks.org/pandas/python-pandas-dataframe-clip/

In [401]:
lines = {}

for column in number_columns:
    q_1 = df_train[column].quantile(0.25)
    q_3 = df_train[column].quantile(0.75)

    iqr = q_3 - q_1

    min_line = q_1 - 1.5 * iqr
    max_line = q_3 + 1.5 * iqr

    lines[column] = (min_line, max_line)

    for data_set in [df_train, df_val, df_test]:
        data_set[column] = data_set[column].clip(min_line, max_line)

**2. Нормализация данных**

Для приведения данных к общему масштабу используем MinMaxScaller: https://scikit-learn.org/stable/modules/generated/sklearn.preprocessing.MinMaxScaler.html

In [402]:
number_columns.append('artists_amount')
number_columns

['duration_ms',
 'danceability',
 'energy',
 'key',
 'loudness',
 'mode',
 'speechiness',
 'acousticness',
 'instrumentalness',
 'liveness',
 'valence',
 'tempo',
 'time_signature',
 'new_explicit',
 'artists_amount']

In [403]:
scaler_obj = MinMaxScaler()
scaler_obj.fit(df_train[number_columns])

,feature_range,"(0, ...)"
,copy,True
,clip,False


In [404]:
for data_set in [df_train, df_val, df_test]:
    data_set[number_columns] = scaler_obj.transform(data_set[number_columns])

In [405]:
df_train

,popularity,duration_ms,explicit,danceability,energy,key,loudness,mode,speechiness,acousticness,...,track_genre_study,track_genre_swedish,track_genre_synth-pop,track_genre_tango,track_genre_techno,track_genre_trance,track_genre_trip-hop,track_genre_turkish,track_genre_world-music,artist_label
54346,0,0.453475,False,0.636766,0.874998,0.181818,0.548894,1.0,0.300313,0.009498,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,12507
46046,56,0.526275,False,0.483942,0.341987,0.090909,0.604274,1.0,0.161129,0.638554,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,2781
9222,4,0.986017,False,0.801772,0.935999,0.363636,0.370692,0.0,0.307210,0.347390,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,9202
59073,31,0.186302,False,0.521595,0.447989,1.000000,0.301479,1.0,0.658307,0.697791,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,10486
58039,9,0.271550,False,0.621262,0.169984,0.545455,0.000000,0.0,0.304075,0.951807,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,14174
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
6265,14,0.511964,False,0.544850,0.938999,0.909091,0.751508,0.0,0.273981,0.036747,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,7159
54886,66,0.366736,False,0.619048,0.724995,0.636364,0.516394,1.0,0.244514,0.160643,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,13034
76820,43,0.640767,False,0.489480,0.994000,0.181818,0.697755,1.0,0.446395,0.000037,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,7378
860,35,0.306315,False,0.444075,0.994000,0.363636,0.549899,0.0,0.477743,0.000446,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,10430


## Формируем итоговые датасеты

In [406]:
df_train.to_csv('df_train.csv')
df_val.to_csv('df_val.csv')
df_test.to_csv('df_test.csv')